# Brain MRI Segmentation — Training on Colab

**Pré-requis :** le stack Docker tourne sur ton PC local (`docker compose up -d --build`) et tu as récupéré l'URL ngrok depuis http://localhost:4040

## 0. Configuration

In [ ]:
# ← Mettre à jour à chaque redémarrage du stack Docker
TRACKING_URI = "https://xxxx.ngrok-free.app"

# Paramètres d'entraînement
UNET_EPOCHS   = 30
UNET_ENCODER  = "resnet34"
UNET_BATCH    = 16
UNET_IMG_SIZE = 256

YOLO_EPOCHS   = 10
YOLO_BATCH    = 16
YOLO_IMG_SIZE = 256

YOLO_DATASET_DIR = "/content/yolo_dataset"

## 1. Setup environnement

In [ ]:
!git clone https://github.com/RomeoCorrec/Brain-MRI-Segmentation.git
%cd Brain-MRI-Segmentation

In [ ]:
# torch et torchvision sont déjà présents sur Colab
!pip install -q \
    segmentation-models-pytorch==0.5.0 \
    ultralytics==8.3.239 \
    albumentations==2.0.8 \
    mlflow==2.14.3 \
    boto3==1.38.0

# Réinstaller numpy pour corriger les incompatibilités binaires avec pandas
!pip install -q -U numpy

# ⚠️ Redémarrer le runtime après cette cellule : Runtime > Restart session
print("\n⚠️  Redémarre le runtime maintenant (Runtime > Restart session), puis continue à la cellule suivante.")

In [ ]:
# Reprendre ici après le redémarrage du runtime
%cd /content/Brain-MRI-Segmentation

import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## 2. Télécharger le dataset TCGA (Kaggle)

In [ ]:
!pip install -q kaggle

# Uploader ton fichier kaggle.json
# (téléchargeable sur https://www.kaggle.com/settings → API → Create New Token)
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d mateuszbuda/lgg-mri-segmentation --unzip -p /content/data
!ls /content/data/

In [ ]:
# Vérifier la structure attendue par build_dataframes()
# Attendu : /content/data/kaggle_3m/TCGA_XXX/img.tif + img_mask.tif
import glob
masks = glob.glob("/content/data/kaggle_3m/*/*_mask.tif")
print(f"{len(masks)} masques trouvés")

DATA_DIR = "/content/data/kaggle_3m"

## 3. Vérifier la connexion MLFlow

In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiments = [e.name for e in client.search_experiments()]
print("Expériences MLFlow :", experiments)
# Attendu : ['Default']  (vide au premier lancement)

## 4. Entraînement UNet

In [ ]:
!python src/train_unet.py \
    --tracking-uri {TRACKING_URI} \
    --data-dir {DATA_DIR} \
    --encoder {UNET_ENCODER} \
    --epochs {UNET_EPOCHS} \
    --batch-size {UNET_BATCH} \
    --image-size {UNET_IMG_SIZE} \
    --output-dir /content/outputs

## 5. Entraînement YOLOv8

In [ ]:
# Convertir les masques TCGA au format YOLO segmentation (polygones normalisés)
# Génère aussi brain_tumor.yaml avec le bon chemin absolu
!python src/prepare_yolo_dataset.py \
    --data-dir {DATA_DIR} \
    --output-dir {YOLO_DATASET_DIR}

In [ ]:
YOLO_YAML = f"{YOLO_DATASET_DIR}/brain_tumor.yaml"

!python src/train_yolo.py \
    --tracking-uri {TRACKING_URI} \
    --data {YOLO_YAML} \
    --epochs {YOLO_EPOCHS} \
    --batch {YOLO_BATCH} \
    --imgsz {YOLO_IMG_SIZE}

## 6. Vérifier les résultats

In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()

print("=== Expériences ===")
for exp in client.search_experiments():
    print(f"  {exp.name}")

print("\n=== Modèles enregistrés ===")
for model in client.search_registered_models():
    latest = model.latest_versions
    for v in latest:
        print(f"  {model.name} — v{v.version} ({v.current_stage})")